# 04 - Nihai Model Karsilastirmasi

Bu defter **model egitmez**. CSP+LDA, EEGNet ve ATCNet kayitli ciktilarini yukler, ayni kosullarda degerlendirildiklerini dogrular ve karsilastirir.

**Bolumler:** arastirma sorusu, degerlendirme protokolu, cikarmalar, model bazli toplu metrikler, katilimci-bazli esli karsilastirma, istatistiksel testler, hesaplama maliyeti, basarisizlik analizi, yorum, sinirliliklar, nihai sonuc.

In [ ]:
# --- Ortak baslangic: proje koku kesfi ve ice aktarmalar ---
import sys, json, warnings
from pathlib import Path

def _find_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "pyproject.toml").exists() and (d / "src" / "cho2017_benchmark").exists():
            return d
    return p

ROOT = _find_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cho2017_benchmark import paths
from cho2017_benchmark.config import resolve_config, dump_resolved_config
from cho2017_benchmark.reproducibility import set_seed, save_environment

set_seed(42)
print("Proje koku:", paths.PROJECT_ROOT)

In [ ]:
QUICK_MODE = False  # Yalnizca hata ayiklama icindir; bilimsel sonuc uretmez.
cfg = resolve_config(quick_mode=QUICK_MODE)
if cfg.quick_mode:
    print("UYARI: QUICK_MODE sonuclari nihai bilimsel sonuc olarak kullanilamaz.")
print("Aktif denek sayisi:", len(cfg.active_subjects()), "| birincil tohum:", cfg.primary_seed)

## Tutarlilik dogrulamasi
Tum modeller ayni split-hash, denek listesi, test deneme kimlikleri ve sinif eslemesini kullanmali; aksi halde durulur.

In [ ]:
from cho2017_benchmark.reporting import comparison as cmp
MODELS = ["csp_lda", "eegnet", "atcnet"]
results = {}
for m in MODELS:
    p = paths.results_dir(m) / "tables" / "subject_metrics.csv"
    if p.exists():
        results[m] = cmp.load_model_results(m)
    else:
        print(f"UYARI: {m} sonuclari yok ({p}). Once {m} defterini calistirin.")
assert len(results) >= 2, "Karsilastirma icin en az iki model gerekir."
cmp.cross_check_consistency(results)
print("Tutarlilik kontrolu GECTI:", list(results))

## Model bazli toplu metrikler (katilimci seviyesinde, tohum-ortalamali)

In [ ]:
from cho2017_benchmark.evaluation.statistics import aggregate_seeds_per_subject, paired_model_comparison, comparison_table
combined = pd.concat([r.subject_metrics.assign(model_name=m) for m, r in results.items()], ignore_index=True)
per_subject = aggregate_seeds_per_subject(combined)
summary = (per_subject.groupby("model_name")[["accuracy", "balanced_accuracy", "macro_f1", "roc_auc"]]
           .agg(["mean", "std", "median"]))
base = paths.results_dir("comparison")
for sub in ["tables", "figures", "reports"]:
    (base / sub).mkdir(parents=True, exist_ok=True)
summary.to_csv(base / "tables" / "model_summary.csv")
summary

## Katilimci-bazli tablolar

In [ ]:
acc_table = cmp.build_subject_accuracy_table(results, "accuracy")
f1_table = cmp.build_subject_accuracy_table(results, "macro_f1")
acc_table.to_csv(base / "tables" / "subject_accuracy.csv", index=False)
f1_table.to_csv(base / "tables" / "subject_macro_f1.csv", index=False)
ranks = cmp.model_rank_per_subject(results, "accuracy"); ranks.to_csv(base / "tables" / "model_rank_per_subject.csv", index=False)
hard = cmp.common_hard_subjects(results, threshold=0.6); print("Ortak zor denekler (<0.6):", hard)
acc_table.head()

## Istatistiksel testler (esli Wilcoxon + bootstrap CI + rank-biserial, Holm duzeltmeli)

In [ ]:
stat_rows = []
for metric in ["accuracy", "macro_f1"]:
    stat_rows.extend(paired_model_comparison(per_subject, metric))
stat_df = comparison_table(stat_rows)
stat_df.to_csv(base / "tables" / "pairwise_statistics.csv", index=False)
stat_df

## Hesaplama maliyeti (parametre, egitim suresi, gecikme)

In [ ]:
cost = (combined[combined["status"] == "ok"]
        .groupby("model_name")[["trainable_parameters", "train_time_seconds", "inference_mean_ms"]]
        .mean())
cost.to_csv(base / "tables" / "cost_comparison.csv")
cost

## Karsilastirma sekilleri

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
data = [per_subject[per_subject.model_name == m]["accuracy"].dropna() for m in results]
ax.boxplot(data, labels=list(results)); ax.axhline(0.5, color="red", ls="--", lw=1)
ax.set_ylabel("dogruluk (katilimci)"); ax.set_title("Modellere gore denek dogruluk dagilimi")
fig.savefig(base / "figures" / "accuracy_distributions.png", dpi=200, bbox_inches="tight")

wide = per_subject.pivot_table(index="subject_id", columns="model_name", values="accuracy")
fig2, ax2 = plt.subplots(figsize=(7, 5))
for _, row in wide.iterrows():
    ax2.plot(list(wide.columns), row.values, color="grey", alpha=0.4, marker="o", ms=3)
ax2.plot(list(wide.columns), wide.mean().values, color="black", marker="s", lw=2, label="ortalama")
ax2.set_ylabel("dogruluk"); ax2.set_title("Esli katilimci dogruluklari"); ax2.legend()
fig2.savefig(base / "figures" / "paired_subject_accuracy.png", dpi=200, bbox_inches="tight")

fig3, ax3 = plt.subplots(figsize=(6, 10))
im = ax3.imshow(wide.values, aspect="auto", cmap="viridis", vmin=0.3, vmax=1.0)
ax3.set_xticks(range(len(wide.columns)), wide.columns); ax3.set_yticks(range(len(wide.index)), wide.index, fontsize=6)
fig3.colorbar(im, ax=ax3); ax3.set_title("denek x model dogruluk")
fig3.savefig(base / "figures" / "subject_model_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

## Yorum, sinirliliklar ve nihai sonuc

In [ ]:
lines = ["# Nihai karsilastirma raporu\n", "## Degerlendirme protokolu",
         "- Denek-bagimli; ayni split manifestosu; test bir kez kullanildi.",
         f"- Katilimci sayisi: {per_subject['subject_id'].nunique()}",
         "", "## Toplu dogruluk (ortalama)"]
for m in results:
    v = per_subject[per_subject.model_name == m]["accuracy"]
    lines.append(f"- {m}: ort={v.mean():.3f}, medyan={v.median():.3f}")
lines += ["", "## Istatistik (accuracy)"]
for r in paired_model_comparison(per_subject, "accuracy"):
    lines.append(f"- {r.comparison}: mean_diff={r.mean_diff:+.3f}, holm_p={r.corrected_p:.3f}, "
                 f"rank_biserial={r.effect_size:+.2f}, anlamli={r.significant}")
lines += ["", "## Sinirliliklar",
          "- Istatistiksel anlamlilik pratik/klinik degeri garanti etmez; etki buyuklugu ve maliyet yorumlanmali.",
          "- Tek oturum; cevrimdisi; deneklerarasi degiskenlik yuksek.",
          "- 'AI ustun' iddiasi yalnizca bir dogruluk/karmasiklik odunlesmesi sunuyor olabilir.",
          "", "## Nihai sonuc",
          "- En yuksek ortalama/medyan dogruluga sahip model ve tutarliligi yukaridaki tablolardan degerlendirilir.",
          "- Hicbir sonuc, defterler gercekten calistirilmadan iddia edilemez."]
(base / "reports" / "final_comparison.md").write_text("\n".join(lines), encoding="utf-8")
print("final_comparison.md yazildi.")

## Generated Files
`results/comparison/tables/*.csv`, `figures/*.png`, `reports/final_comparison.md`.

In [ ]:
expected = [base / "tables" / "model_summary.csv", base / "tables" / "pairwise_statistics.csv",
            base / "reports" / "final_comparison.md"]
missing = [str(p) for p in expected if not p.exists()]
assert not missing, f"Beklenen karsilastirma ciktilari eksik: {missing}"
print("Tum beklenen karsilastirma ciktilari mevcut.")